# Running Mamba on Kaggle GPUs: A Workaround
# Works on GPU T4x2

Mamba models are powerful sequence models, but running them on Kaggle can be tricky due to GPU and library constraints.  
In this notebook, we’ll walk through a step-by-step method to run Mamba efficiently on Kaggle.

- Directly Installing mamba_ssm via pip install gives an run time import error of selective scan cuda
- I haven't fully understood the reason behind it, but after going through som github issues on mamba and causal conv1d, found a way around by downgrading to pytorch 2.4.
- This notebook works only on GPU T4x2 not P100 due to the arch type 6.0.

In [1]:
# The pytorch version 2.4 is mostly tested for mamba_ssm Library so Downgrading default Pytorch version
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 6.1 MB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.4 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 72.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 45.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 5.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128

In [2]:
# Based on the GPU requriments of Kaggle and Pytorch version, downloading the appropriate Wheel files
!wget https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.2/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

!wget https://github.com/state-spaces/mamba/releases/download/v2.2.5/mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

--2026-04-20 17:04:31--  https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.2/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/726718359/604a2e41-ef67-4125-bb18-e5c1bfafc2be?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-20T18%3A03%3A27Z&rscd=attachment%3B+filename%3Dcausal_conv1d-1.5.2%2Bcu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-20T17%3A02%3A52Z&ske=2026-04-20T18%3A03%3A27Z&sks=b&skv=2018-11-09&sig=gY5gUeN4jhyjcAYdsVofFPS%2BRM5MatBUM873HK67Ssk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidX

In [3]:
# Installing the Wheel files
!pip install /kaggle/working/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!pip install /kaggle/working/mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

Processing ./causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
Processing ./mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl


In [4]:
#Testing the Mamba Libray
import torch

from mamba_ssm import Mamba

batch, length, dim = 2, 64, 16
x = torch.randn(batch, length, dim).to("cuda")
model = Mamba(
    # This module uses roughly 3 * expand * d_model^2 parameters
    d_model=dim, # Model dimension d_model
    d_state=16,  # SSM state expansion factor
    d_conv=4,    # Local convolution width
    expand=2,    # Block expansion factor
).to("cuda")
y = model(x)
assert y.shape == x.shape

In [10]:
import torch
import torch.nn as nn
import timm
from mamba_ssm import Mamba

class CoAtMambaMTL(nn.Module):
    def __init__(
        self, 
        num_clinical_features, 
        d_model=768,       
        mamba_d_state=16,  
        mamba_d_conv=4,    
        mamba_expand=2,    
        freeze_coatnet=False
    ):
        super(CoAtMambaMTL, self).__init__()
        
        # 1. Vision Modality
        self.coatnet = timm.create_model('coatnet_0_rw_224.sw_in1k', pretrained=True, num_classes=0)
        coatnet_out_dim = self.coatnet.num_features 
        
        if freeze_coatnet:
            for param in self.coatnet.parameters():
                param.requires_grad = False
                
        self.vision_proj = nn.Linear(coatnet_out_dim, d_model) if coatnet_out_dim != d_model else nn.Identity()

        # 2. Clinical Modality
        self.clinical_mlp = nn.Sequential(
            nn.Linear(num_clinical_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, d_model) 
        )

        # --- THE STABILIZER: LayerNorm ---
        # This ensures Clinical and Vision tokens are on the same scale (Mean 0, Var 1)
        self.norm = nn.LayerNorm(d_model)

        # 3. Mamba Sequencer
        self.mamba_block = Mamba(
            d_model=d_model,
            d_state=mamba_d_state,
            d_conv=mamba_d_conv,
            expand=mamba_expand,
        )

        # 4. Multi-Task Heads
        self.head_rec = nn.Linear(d_model, 1) 
        self.head_lnm = nn.Linear(d_model, 1) 
        self.head_td = nn.Linear(d_model, 1)  
        self.head_ti = nn.Linear(d_model, 1)  
        self.head_ce = nn.Linear(d_model, 1)  
        self.head_pi = nn.Linear(d_model, 1)  

    def forward(self, images, clinical_data):
        B, num_images, C, H, W = images.shape
        
        # Process Vision
        images_reshaped = images.view(B * num_images, C, H, W)
        vision_features = self.coatnet(images_reshaped)
        vision_features = self.vision_proj(vision_features)
        pathology_tokens = vision_features.view(B, num_images, -1)
        
        # Process Clinical
        clinical_token = self.clinical_mlp(clinical_data).unsqueeze(1)
        
        # Combine & Normalize
        sequence = torch.cat([clinical_token, pathology_tokens], dim=1)
        # Apply normalization BEFORE Mamba to balance modalities
        sequence = self.norm(sequence) 
        
        # Pass through Mamba
        mamba_out = self.mamba_block(sequence)
        
        # --- THE FIX: Mean Pooling ---
        # Average across all 7 tokens (Clinical + 6 Images) so index 0 isn't forgotten
        final_state = mamba_out.mean(dim=1) 
        
        # Logits
        return {
            'REC': self.head_rec(final_state).squeeze(1),
            'LNM': self.head_lnm(final_state).squeeze(1),
            'TD': self.head_td(final_state).squeeze(1),
            'TI': self.head_ti(final_state).squeeze(1),
            'CE': self.head_ce(final_state).squeeze(1),
            'PI': self.head_pi(final_state).squeeze(1),
        }

# ==========================================
# TEST BLOCK
# ==========================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size, num_clinical_vars, d_model = 2, 12, 768
    
    mock_images = torch.randn(batch_size, 6, 3, 224, 224).to(device) 
    mock_clinical = torch.randn(batch_size, num_clinical_vars).to(device)
    
    model = CoAtMambaMTL(num_clinical_features=num_clinical_vars, d_model=d_model).to(device)
    
    # 1. Verification of Clinical Sensitivity
    model.eval()
    with torch.no_grad():
        out_A = model(mock_images, mock_clinical)
        out_B = model(mock_images, torch.zeros_like(mock_clinical))
        
    diff = (out_A['REC'] - out_B['REC']).abs().mean().item()
    print(f"--- Modality Sensitivity Check ---")
    print(f"Clinical Token Influence (Diff): {diff:.8f}")
    
    if diff > 1e-4:
        print("✅ SUCCESS: Clinical data is significantly influencing the output.")
    else:
        print("⚠️ WARNING: Clinical signal is still weak. Consider increasing MLP width.")
        
    print("\n✅ All shapes verified at 224x224 resolution.")

--- Modality Sensitivity Check ---
Clinical Token Influence (Diff): 0.00141558
✅ SUCCESS: Clinical data is significantly influencing the output.

✅ All shapes verified at 224x224 resolution.


In [7]:
import math

# Pass random tensors
preds = model(torch.randn(2, 6, 3, 224, 224).to(device), torch.randn(2, num_clinical_vars).to(device))
labels = torch.randint(0, 2, (2,)).float().to(device)

criterion = torch.nn.BCEWithLogitsLoss()
initial_loss = criterion(preds['REC'], labels).item()

print(f"Calculated Initial Loss: {initial_loss:.4f}")
print(f"Expected Initial Loss: {0.693:.4f}")

# If your initial loss is something crazy like 5.5 or 0.01, your final linear layers 
# have a severe initialization bias and the model will struggle to start training.

Calculated Initial Loss: 0.6935
Expected Initial Loss: 0.6930


In [8]:
model.train() # Ensure BatchNorm/Dropout are active

# --- UPDATED: Safe Batch Size and Target Resolution ---
batch_size = 2 # Reduced from 16 to survive Kaggle's VRAM limits

# 1. Create the dummy data (6 images per patient at 512x512 resolution)
dummy_images = torch.randn(batch_size, 6, 3, 224, 224).to(device)
dummy_clinical = torch.randn(batch_size, num_clinical_vars).to(device)

print(f"Pushing {batch_size} patients (12 high-res images) through the model...")

# 2. ACTUALLY call the model! This triggers the forward() function and your debug print.
_ = model(dummy_images, dummy_clinical)

print("Forward pass complete!")

Pushing 2 patients (12 high-res images) through the model...
Forward pass complete!
